# 🏋️ Week 2: Classical ML Practice Exercises (Enhanced)

This notebook contains comprehensive practice exercises with **detailed explanations** of:
- **WHAT** each algorithm/metric is
- **WHY** it works the way it does (mathematical intuition)
- **HOW** it operates under the hood
- **WHEN** to use it in real-world applications

---

In [ ]:
# Standard imports for all exercises
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
np.random.seed(42)
print("✅ Imports ready!")

---

## �� Exercise 1: Implement Classification Metrics from Scratch

### What is a Confusion Matrix?
A **confusion matrix** is a table that describes the performance of a classification model by comparing predicted vs actual values.

```
                    Predicted
                 Positive  Negative
Actual Positive    TP        FN
       Negative    FP        TN
```

### Understanding TP, TN, FP, FN
| Term | Meaning | Example (Spam Detection) |
|------|---------|-------------------------|
| **TP** (True Positive) | Correctly predicted positive | Spam email → predicted as spam ✓ |
| **TN** (True Negative) | Correctly predicted negative | Normal email → predicted as normal ✓ |
| **FP** (False Positive) | Incorrectly predicted positive | Normal email → predicted as spam ✗ |
| **FN** (False Negative) | Incorrectly predicted negative | Spam email → predicted as normal ✗ |

### Key Metrics Derived from Confusion Matrix

#### 1. Accuracy
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
- **What**: Proportion of correct predictions
- **When to use**: Balanced classes
- **When NOT to use**: Imbalanced data (99% accuracy can be useless!)

#### 2. Precision
$$\text{Precision} = \frac{TP}{TP + FP}$$
- **What**: Of all predicted positives, how many are actually positive?
- **When to use**: When **false positives are costly** (spam filter - don't mark good emails as spam)
- **Intuition**: "When I say YES, am I right?"

#### 3. Recall (Sensitivity, True Positive Rate)
$$\text{Recall} = \frac{TP}{TP + FN}$$
- **What**: Of all actual positives, how many did we catch?
- **When to use**: When **false negatives are costly** (disease detection - don't miss sick patients)
- **Intuition**: "Did I find all the positives?"

#### 4. F1-Score
$$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
- **What**: Harmonic mean of precision and recall
- **When to use**: When you need balance between precision and recall
- **Why harmonic mean?**: Penalizes extreme values (if one metric is 0, F1 is 0)

### Real-World Trade-offs

| Application | Priority | Why |
|-------------|----------|-----|
| Cancer detection | High Recall | Missing cancer (FN) is dangerous |
| Spam filter | High Precision | Blocking good email (FP) is annoying |
| Fraud detection | Balance | Both missing fraud and false alarms are costly |

---

In [ ]:
def calculate_metrics(y_true: list, y_pred: list) -> dict:
    """
    Calculate classification metrics from scratch (no sklearn!).
    
    Algorithm:
    1. Count TP, TN, FP, FN by comparing each (true, pred) pair
    2. Apply formulas for each metric
    3. Handle edge cases (division by zero)
    
    Args:
        y_true: Ground truth labels (0 or 1)
        y_pred: Predicted labels (0 or 1)
    
    Returns:
        Dictionary with accuracy, precision, recall, f1
    """
    # YOUR CODE HERE
    # Count TP, TN, FP, FN first
    pass


# Test
y_true = [1, 0, 1, 1, 0, 1, 0, 0, 1, 1]
y_pred = [1, 0, 0, 1, 0, 1, 1, 0, 1, 0]

metrics = calculate_metrics(y_true, y_pred)
assert abs(metrics['accuracy'] - 0.7) < 0.01, f"Accuracy wrong: {metrics['accuracy']}"
assert abs(metrics['precision'] - 0.8) < 0.01, f"Precision wrong: {metrics['precision']}"
assert abs(metrics['recall'] - 0.67) < 0.01, f"Recall wrong: {metrics['recall']}"
print("✅ All tests passed!")
print(f"Metrics: {metrics}")

In [ ]:
# Solution with detailed explanation
def calculate_metrics_solution(y_true: list, y_pred: list) -> dict:
    # Step 1: Count confusion matrix elements
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
    
    print(f"Confusion Matrix Elements:")
    print(f"  TP={tp}, TN={tn}, FP={fp}, FN={fn}")
    
    # Step 2: Calculate metrics with edge case handling
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0  # Avoid div by zero
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'accuracy': round(accuracy, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1': round(f1, 4)
    }

# Test with our example
print("\nMetrics calculation:")
result = calculate_metrics_solution(y_true, y_pred)
print(f"\nFinal metrics: {result}")

# Verify with sklearn
print(f"\nsklearn verification:")
print(f"  Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"  Precision: {precision_score(y_true, y_pred):.4f}")
print(f"  Recall: {recall_score(y_true, y_pred):.4f}")
print(f"  F1: {f1_score(y_true, y_pred):.4f}")

---

## 📚 Exercise 2: Model Selection with Cross-Validation

### What is Cross-Validation?
**Cross-validation** is a technique to evaluate how well a model generalizes to unseen data by splitting the data into multiple train/test folds.

### K-Fold Cross-Validation
```
5-Fold CV Example:

Fold 1: [TEST] [train] [train] [train] [train]
Fold 2: [train] [TEST] [train] [train] [train]
Fold 3: [train] [train] [TEST] [train] [train]
Fold 4: [train] [train] [train] [TEST] [train]
Fold 5: [train] [train] [train] [train] [TEST]

Final Score = Average of all 5 folds
```

### Why Use Cross-Validation?
1. **More Reliable Estimate**: Uses all data for both training and testing
2. **Reduces Variance**: Single train/test split can be "lucky" or "unlucky"
3. **Detects Overfitting**: Large std across folds = model is unstable

### Cross-Validation Variants
| Variant | Description | Use When |
|---------|-------------|----------|
| K-Fold | Standard k splits | General purpose |
| Stratified K-Fold | Maintains class ratio | Imbalanced classification |
| Leave-One-Out | K = n (one sample test) | Very small datasets |
| Time Series Split | Forward-only splits | Sequential data |

### How to Compare Models
```python
# Don't just compare means!
Model A: 0.85 (+/- 0.02)  # Stable
Model B: 0.86 (+/- 0.10)  # Unstable

# Model A might be better for production despite lower mean
```

### Real-World Application
- **Hyperparameter Tuning**: Use CV to find best hyperparameters
- **Model Selection**: Compare different algorithms fairly
- **Feature Selection**: Evaluate impact of features

---

In [ ]:
def find_best_model(X, y, models: dict, metric='accuracy', cv=5):
    """
    Compare multiple models using cross-validation.
    
    Algorithm:
    1. For each model:
       - Run K-fold cross-validation
       - Record mean and std of scores
    2. Find model with highest mean score
    3. Return best model and all results
    
    Args:
        X: Feature matrix
        y: Target vector
        models: Dict of {name: model_instance}
        metric: Scoring metric (accuracy, f1, roc_auc, etc.)
        cv: Number of cross-validation folds
    
    Returns:
        Tuple of (best_model_name, best_model_instance, all_scores_dict)
    """
    # YOUR CODE HERE
    pass


# Test with synthetic data
X, y = make_classification(n_samples=500, n_features=10, n_informative=5, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42)
}

best_name, best_model, all_scores = find_best_model(X, y, models)
print(f"Best model: {best_name}")

In [ ]:
# Solution with detailed comparison
def find_best_model_solution(X, y, models: dict, metric='accuracy', cv=5):
    scores = {}
    
    print(f"Comparing {len(models)} models with {cv}-fold CV...\n")
    
    for name, model in models.items():
        # Run cross-validation
        cv_scores = cross_val_score(model, X, y, cv=cv, scoring=metric)
        
        scores[name] = {
            'mean': cv_scores.mean(),
            'std': cv_scores.std(),
            'scores': cv_scores,
            'min': cv_scores.min(),
            'max': cv_scores.max()
        }
        
        print(f"{name}:")
        print(f"  Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
        print(f"  Range: [{cv_scores.min():.4f}, {cv_scores.max():.4f}]")
        print(f"  All folds: {cv_scores.round(4)}\n")
    
    # Find best model
    best_name = max(scores, key=lambda x: scores[x]['mean'])
    best_model = models[best_name]
    
    return best_name, best_model, scores

# Run comparison
best_name, best_model, all_scores = find_best_model_solution(X, y, models)

print(f"\n🏆 Best Model: {best_name}")

# Visualize results
fig, ax = plt.subplots(figsize=(10, 5))
names = list(all_scores.keys())
means = [all_scores[n]['mean'] for n in names]
stds = [all_scores[n]['std'] for n in names]

bars = ax.bar(names, means, yerr=stds, capsize=5, alpha=0.7)
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison with Cross-Validation')
ax.set_ylim(0.7, 1.0)

# Highlight best
bars[names.index(best_name)].set_color('green')
plt.tight_layout()
plt.show()

---

## 📚 Exercise 3: Overfitting Detection and Diagnosis

### What is Overfitting?
**Overfitting** occurs when a model learns the training data too well, including noise and outliers, leading to poor generalization on new data.

### Visual Understanding
```
Training Accuracy    |████████████████████| 100%
Test Accuracy        |████████████        | 60%
                     ↑ BIG GAP = OVERFITTING
```

### Bias-Variance Tradeoff
| Condition | Bias | Variance | Training Error | Test Error | Diagnosis |
|-----------|------|----------|----------------|------------|-----------|
| Underfitting | High | Low | High | High | Model too simple |
| Optimal | Low | Low | Low | Low | Just right |
| Overfitting | Low | High | Very Low | High | Model too complex |

### How to Detect Overfitting
1. **Large Train-Test Gap**: Train accuracy >> Test accuracy
2. **High CV Variance**: Scores vary widely across folds
3. **Learning Curves**: Test error increases while train error decreases

### How to Fix Overfitting
| Technique | How It Helps | Example |
|-----------|--------------|--------|
| Regularization | Penalizes complex models | L1/L2 penalty |
| Early Stopping | Stop before overfitting | Stop training when val loss increases |
| Dropout | Random neuron deactivation | Neural networks |
| More Data | More examples = harder to memorize | Data augmentation |
| Feature Selection | Remove noisy features | PCA, feature importance |
| Simpler Model | Fewer parameters | Reduce depth, fewer trees |
| Ensemble | Average multiple models | Random Forest, Bagging |

### Decision Tree Specific Parameters
| Parameter | Effect on Overfitting |
|-----------|----------------------|
| `max_depth` | Lower = less overfitting |
| `min_samples_split` | Higher = less overfitting |
| `min_samples_leaf` | Higher = less overfitting |
| `max_features` | Lower = less overfitting |

---

In [ ]:
def diagnose_model(model, X_train, y_train, X_test, y_test, threshold=0.1):
    """
    Diagnose if a model is overfitting, underfitting, or well-fitted.
    
    Algorithm:
    1. Calculate train and test accuracy
    2. Calculate the gap (train - test)
    3. Classify:
       - Overfitting: high train, low test (gap > threshold)
       - Underfitting: low train, low test (both < 0.7)
       - Good fit: small gap, both reasonable
    
    Args:
        model: Trained sklearn model
        X_train, y_train: Training data
        X_test, y_test: Test data
        threshold: Max acceptable gap
    
    Returns:
        Dictionary with diagnosis
    """
    # YOUR CODE HERE
    pass


# Create data
X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Test with overfitting model
overfit_model = DecisionTreeClassifier(max_depth=None)  # No regularization
overfit_model.fit(X_train, y_train)

result = diagnose_model(overfit_model, X_train, y_train, X_test, y_test)
print(f"Results: {result}")
assert result['diagnosis'] == 'overfitting', "Should detect overfitting!"
print("✅ Correctly detected overfitting!")

In [ ]:
# Solution with comprehensive diagnosis
def diagnose_model_solution(model, X_train, y_train, X_test, y_test, threshold=0.1):
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    gap = train_acc - test_acc
    
    # Diagnosis logic
    if gap > threshold:
        diagnosis = 'overfitting'
        recommendation = 'Add regularization, reduce model complexity, or get more data'
    elif train_acc < 0.7 and test_acc < 0.7:
        diagnosis = 'underfitting'
        recommendation = 'Increase model complexity, add features, or train longer'
    else:
        diagnosis = 'good_fit'
        recommendation = 'Model looks good! Consider ensemble for slight improvement'
    
    return {
        'train_accuracy': round(train_acc, 4),
        'test_accuracy': round(test_acc, 4),
        'gap': round(gap, 4),
        'diagnosis': diagnosis,
        'recommendation': recommendation
    }

# Compare different models
print("=" * 60)
print("Model Diagnosis Comparison")
print("=" * 60)

# Overfitting model
print("\n1. Deep Decision Tree (max_depth=None):")
overfit_tree = DecisionTreeClassifier(max_depth=None)
overfit_tree.fit(X_train, y_train)
result = diagnose_model_solution(overfit_tree, X_train, y_train, X_test, y_test)
for k, v in result.items():
    print(f"   {k}: {v}")

# Regularized model
print("\n2. Shallow Decision Tree (max_depth=3):")
good_tree = DecisionTreeClassifier(max_depth=3)
good_tree.fit(X_train, y_train)
result = diagnose_model_solution(good_tree, X_train, y_train, X_test, y_test)
for k, v in result.items():
    print(f"   {k}: {v}")

# Ensemble model
print("\n3. Random Forest (n_estimators=100):")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
result = diagnose_model_solution(rf, X_train, y_train, X_test, y_test)
for k, v in result.items():
    print(f"   {k}: {v}")

---

## 📚 Exercise 4: Feature Importance Analysis

### What is Feature Importance?
**Feature importance** tells us how much each feature contributes to the model's predictions. It's crucial for:
- Understanding model behavior
- Feature selection
- Debugging unexpected predictions

### How Tree-Based Models Calculate Importance

#### Gini Importance (Default in sklearn)
For each feature, measure the total reduction in impurity (Gini or entropy) across all splits:

$$\text{Importance}(f) = \sum_{\text{nodes using } f} \frac{n_{\text{node}}}{n_{\text{total}}} \times \Delta \text{Impurity}$$

#### Permutation Importance
Shuffle one feature, measure accuracy drop:
1. Train model, record baseline accuracy
2. Shuffle feature X, record new accuracy
3. Importance = baseline - shuffled accuracy

### Why Gini Importance Can Be Misleading
- **Biased toward high-cardinality features**: More unique values = more split opportunities
- **Doesn't account for feature correlation**: Correlated features share importance
- **Only works for tree models**: Not generalizable

### Interpretation Guidelines
| Importance Value | Interpretation |
|-----------------|----------------|
| > 0.1 | Highly important |
| 0.05 - 0.1 | Moderately important |
| < 0.05 | Less important |
| ~0 | Possibly removable |

### Real-World Application
- **Credit Risk**: Which factors most affect default probability?
- **Churn Prediction**: What drives customer departure?
- **Medical Diagnosis**: Which symptoms are most predictive?

---

In [ ]:
def analyze_feature_importance(model, feature_names: list, top_n: int = 5) -> list:
    """
    Analyze and visualize feature importances from tree-based models.
    
    Algorithm:
    1. Extract feature_importances_ from model
    2. Pair with feature names
    3. Sort by importance descending
    4. Return top N
    
    Args:
        model: Trained model with feature_importances_ attribute
        feature_names: List of feature names
        top_n: Number of top features to return
    
    Returns:
        List of (feature_name, importance) tuples
    """
    # YOUR CODE HERE
    pass


# Test
X, y = make_classification(
    n_samples=500, 
    n_features=10, 
    n_informative=5,  # Only 5 features are actually useful!
    n_redundant=2,
    random_state=42
)
feature_names = [f'feature_{i}' for i in range(10)]

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

top_features = analyze_feature_importance(rf, feature_names, top_n=5)
print("Top 5 Features:")
for name, importance in top_features:
    print(f"  {name}: {importance:.4f}")

In [ ]:
# Solution with visualization and interpretation
def analyze_feature_importance_solution(model, feature_names: list, top_n: int = 5) -> list:
    # Get importances
    importances = model.feature_importances_
    
    # Create pairs and sort
    feature_importance = list(zip(feature_names, importances))
    sorted_features = sorted(feature_importance, key=lambda x: x[1], reverse=True)
    
    return sorted_features[:top_n]

# Full analysis
all_features = analyze_feature_importance_solution(rf, feature_names, top_n=10)

# Categorize features
print("Feature Importance Analysis:")
print("=" * 50)
for name, importance in all_features:
    # Categorize
    if importance > 0.1:
        category = "🔴 High"
    elif importance > 0.05:
        category = "🟡 Medium"
    else:
        category = "🟢 Low"
    
    bar = "█" * int(importance * 50)
    print(f"{name:15} | {bar:25} | {importance:.4f} {category}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
names, importances = zip(*all_features)
colors = ['green' if i < 5 else 'gray' for i in range(len(names))]
ax1.barh(range(len(names)), importances, color=colors)
ax1.set_yticks(range(len(names)))
ax1.set_yticklabels(names)
ax1.set_xlabel('Importance')
ax1.set_title('Feature Importances (Top 5 in green)')
ax1.invert_yaxis()

# Cumulative importance
cumsum = np.cumsum(importances)
ax2.plot(range(1, len(names)+1), cumsum, 'b-o', markersize=8)
ax2.axhline(y=0.9, color='r', linestyle='--', label='90% threshold')
ax2.fill_between(range(1, len(names)+1), cumsum, alpha=0.3)
ax2.set_xlabel('Number of Features')
ax2.set_ylabel('Cumulative Importance')
ax2.set_title('Cumulative Feature Importance')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find minimum features for 90% importance
for i, cum in enumerate(cumsum):
    if cum >= 0.9:
        print(f"\n💡 Insight: Top {i+1} features capture {cum:.1%} of importance")
        break

---

## 📚 Exercise 5: ROC Curve and Threshold Optimization

### What is an ROC Curve?
**ROC (Receiver Operating Characteristic)** curve plots the trade-off between True Positive Rate (Recall) and False Positive Rate at various classification thresholds.

### Understanding the Axes
$$\text{TPR (Recall)} = \frac{TP}{TP + FN}$$ → "Of all positives, how many did we catch?"

$$\text{FPR} = \frac{FP}{FP + TN}$$ → "Of all negatives, how many did we falsely flag?"

### What is AUC (Area Under Curve)?
- **AUC = 1.0**: Perfect classifier
- **AUC = 0.5**: Random guess (diagonal line)
- **AUC < 0.5**: Worse than random (invert predictions!)

### AUC Interpretation
| AUC Range | Quality |
|-----------|--------|
| 0.9 - 1.0 | Excellent |
| 0.8 - 0.9 | Good |
| 0.7 - 0.8 | Fair |
| 0.6 - 0.7 | Poor |
| 0.5 - 0.6 | Fail |

### Why ROC/AUC Matters
1. **Threshold Independent**: Evaluates model across all thresholds
2. **Class Imbalance Robust**: Better than accuracy for imbalanced data
3. **Model Comparison**: Compare models without choosing threshold

### Threshold Optimization
Default threshold is 0.5, but optimal threshold depends on:
- **Cost of False Positives vs False Negatives**
- **Business requirements**

Common methods:
- **Youden's J**: Maximize TPR - FPR
- **Precision-Recall balance**: Find intersection
- **Cost-based**: Minimize total business cost

---

In [ ]:
def find_optimal_threshold(y_true, y_proba, method='youden'):
    """
    Find the optimal classification threshold.
    
    Methods:
    - 'youden': Maximize TPR - FPR (Youden's J statistic)
    - 'f1': Maximize F1 score
    
    Args:
        y_true: True labels
        y_proba: Predicted probabilities for positive class
        method: Optimization method
    
    Returns:
        Dict with optimal_threshold, tpr, fpr, and metric value
    """
    # YOUR CODE HERE
    # Hint: Use roc_curve to get thresholds, then iterate
    pass


# Test
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

result = find_optimal_threshold(y_test, y_proba)
print(f"Optimal threshold: {result['optimal_threshold']:.4f}")

In [ ]:
# Solution with comprehensive analysis
def find_optimal_threshold_solution(y_true, y_proba, method='youden'):
    # Get ROC curve data
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    
    if method == 'youden':
        # Youden's J = TPR - FPR (maximize)
        j_scores = tpr - fpr
        best_idx = np.argmax(j_scores)
        metric_name = "Youden's J"
        metric_value = j_scores[best_idx]
    elif method == 'f1':
        # Try each threshold and calculate F1
        f1_scores = []
        for thresh in thresholds:
            y_pred = (y_proba >= thresh).astype(int)
            f1_scores.append(f1_score(y_true, y_pred))
        best_idx = np.argmax(f1_scores)
        metric_name = 'F1 Score'
        metric_value = f1_scores[best_idx]
    
    return {
        'optimal_threshold': thresholds[best_idx],
        'tpr': tpr[best_idx],
        'fpr': fpr[best_idx],
        'metric_name': metric_name,
        'metric_value': metric_value
    }

# Full analysis
result_youden = find_optimal_threshold_solution(y_test, y_proba, 'youden')
result_f1 = find_optimal_threshold_solution(y_test, y_proba, 'f1')

print("Threshold Optimization Results:")
print("=" * 50)
print(f"\nYouden's Method:")
print(f"  Optimal Threshold: {result_youden['optimal_threshold']:.4f}")
print(f"  TPR: {result_youden['tpr']:.4f}, FPR: {result_youden['fpr']:.4f}")
print(f"  J Score: {result_youden['metric_value']:.4f}")

print(f"\nF1 Maximization:")
print(f"  Optimal Threshold: {result_f1['optimal_threshold']:.4f}")
print(f"  F1 Score: {result_f1['metric_value']:.4f}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
ax1.scatter([result_youden['fpr']], [result_youden['tpr']], 
           color='red', s=100, zorder=5, label=f"Optimal (Youden's J)")
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve with Optimal Threshold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Threshold vs Metrics
precisions = []
recalls = []
f1s = []
for thresh in thresholds:
    y_pred = (y_proba >= thresh).astype(int)
    precisions.append(precision_score(y_true, y_pred, zero_division=0))
    recalls.append(recall_score(y_true, y_pred, zero_division=0))
    f1s.append(f1_score(y_true, y_pred, zero_division=0))

ax2.plot(thresholds, precisions, 'g-', label='Precision')
ax2.plot(thresholds, recalls, 'b-', label='Recall')
ax2.plot(thresholds, f1s, 'r-', linewidth=2, label='F1')
ax2.axvline(x=result_f1['optimal_threshold'], color='red', linestyle='--', 
           label=f"Optimal F1 = {result_f1['optimal_threshold']:.3f}")
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Score')
ax2.set_title('Precision, Recall, F1 vs Threshold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare default vs optimal
print("\nComparison: Default (0.5) vs Optimal Threshold")
print("-" * 50)
for thresh, name in [(0.5, 'Default (0.5)'), (result_f1['optimal_threshold'], 'Optimal')]:
    y_pred = (y_proba >= thresh).astype(int)
    print(f"\n{name}:")
    print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"  Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"  F1:        {f1_score(y_test, y_pred):.4f}")

---

## 📚 Exercise 6: Learning Curves - Diagnosing with Data Size

### What are Learning Curves?
**Learning curves** show how model performance changes as training data increases. They help diagnose:
- Whether more data would help
- Bias vs variance issues
- Optimal training set size

### How to Read Learning Curves

```
OVERFITTING:              UNDERFITTING:            GOOD FIT:
│                         │                        │     
│╭─────────train          │──────────train        │╭────train
││                        │──────────test         │├────test
││    ╭───test            │                        ││
││   ╱                    │                        ││
│╰──╱                     │                        ││
└────────────────>        └─────────────────>      └───────────>
  Training Size             Training Size           Training Size

Gap stays large           Both plateau low         Converge high
→ Need regularization     → Need complex model     → Model is good
```

### Key Insights
| Pattern | Diagnosis | Action |
|---------|-----------|--------|
| Large gap, train high, test low | Overfitting | More data, regularization |
| Small gap, both low | Underfitting | More complex model |
| Curves still converging | Need more data | Collect more data |
| Curves plateaued | More data won't help | Focus on features/model |

---

In [ ]:
from sklearn.model_selection import learning_curve

def plot_learning_curves(model, X, y, title='Learning Curves', cv=5):
    """
    Generate and plot learning curves.
    
    Shows how training and validation scores change with training set size.
    
    Args:
        model: sklearn estimator
        X: Feature matrix
        y: Target vector
        title: Plot title
        cv: Cross-validation folds
    """
    # YOUR CODE HERE
    # Use sklearn's learning_curve function
    pass

In [ ]:
# Solution with comprehensive analysis
def plot_learning_curves_solution(model, X, y, title='Learning Curves', cv=5):
    # Generate learning curve data
    train_sizes = np.linspace(0.1, 1.0, 10)
    train_sizes_abs, train_scores, val_scores = learning_curve(
        model, X, y, 
        train_sizes=train_sizes, 
        cv=cv, 
        scoring='accuracy',
        n_jobs=-1
    )
    
    # Calculate mean and std
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    plt.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std, alpha=0.1, color='orange')
    plt.plot(train_sizes_abs, train_mean, 'o-', color='blue', label='Training score')
    plt.plot(train_sizes_abs, val_mean, 'o-', color='orange', label='Validation score')
    
    plt.xlabel('Training Set Size')
    plt.ylabel('Accuracy')
    plt.title(title)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    
    # Diagnosis
    final_gap = train_mean[-1] - val_mean[-1]
    final_train = train_mean[-1]
    final_val = val_mean[-1]
    
    return {
        'final_train_score': final_train,
        'final_val_score': final_val,
        'gap': final_gap,
        'train_sizes': train_sizes_abs,
        'train_scores': train_mean,
        'val_scores': val_mean
    }

# Compare overfitting vs good model
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overfitting model
plt.sca(axes[0])
overfit = DecisionTreeClassifier(max_depth=None)
result1 = plot_learning_curves_solution(overfit, X, y, 'Overfitting Model (Deep Tree)')

# Good model
plt.sca(axes[1])
good = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
result2 = plot_learning_curves_solution(good, X, y, 'Good Model (Random Forest)')

plt.tight_layout()
plt.show()

print("\nDiagnosis:")
print(f"Deep Tree - Gap: {result1['gap']:.4f} (Overfitting)")
print(f"Random Forest - Gap: {result2['gap']:.4f} (Good Fit)")

---

## 📋 Week 2 Practice Summary

### Concepts Covered

| Topic | Key Insight | Interview Tip |
|-------|-------------|---------------|
| Confusion Matrix | TP/FP/TN/FN foundation | Draw it out! |
| Precision vs Recall | Trade-off depends on cost | Know when to prioritize each |
| Cross-Validation | Reduces variance in estimates | Always use for model selection |
| Overfitting | Train >> Test = problem | Check learning curves |
| Feature Importance | Which features matter | Use for feature selection |
| ROC/AUC | Threshold-independent | Great for imbalanced data |
| Learning Curves | Diagnose with data size | More data vs better model |

### Exercises Completed
- [ ] Classification Metrics from Scratch
- [ ] Model Selection with CV
- [ ] Overfitting Detection
- [ ] Feature Importance Analysis
- [ ] ROC and Threshold Optimization
- [ ] Learning Curves

### Interview Formulas to Memorize
- **Precision** = TP / (TP + FP)
- **Recall** = TP / (TP + FN)
- **F1** = 2 × (P × R) / (P + R)
- **Accuracy** = (TP + TN) / Total
- **FPR** = FP / (FP + TN)

---
**Ready for Week 3: Preprocessing & Feature Engineering!** 🚀